# H2 plasma flux diagnostics example

This notebook demonstrates the tools in `flux_diagnostics.py` on a DC/RF H2
discharge whose diagnostics were collected into a directory named `diags`
(matching `Analysis()`'s own default directory and this repo's `main.py`
default output folder name).

It's a template: run it once your own `diags` run has produced output
(`diagnostic_times.dat`, `eadf_*`, `time_averaged_*`). Every cell below has
been validated against an equivalent real dataset, so the only thing that
should need to change for a different case is the `directory` and
`pressure_torr` in the second cell.

In [ ]:
import json

import matplotlib.pyplot as plt

from analysis import Analysis
import flux_diagnostics as fd

## 1. Load the diagnostics

`summarize_anode_fluxes` needs the gas pressure (for the H-radical diffusion
solve) -- everything else (species names, the anode wall convention, the
boundary-scraping undercount correction) has a sensible default. See
`flux_diagnostics.py`'s module docstring for what those defaults assume and
why, before reusing them on a differently-wired input file.

In [ ]:
diag = Analysis('diags', quiet_startup=True)

PRESSURE_TORR = 28  # gas pressure this simulation was run at

## 2. Ion, electron, and H-radical flux at the anode

`summarize_anode_fluxes` is the one-call entry point: ion and electron flux
from the wall EDFs (corrected for the anode boundary-scraping undercount and
the EDF `dz` unit convention -- see the module docstring), and H-radical
flux from a steady-state reaction-rate diffusion solve.

In [ ]:
summary = fd.summarize_anode_fluxes(diag, pressure_torr=PRESSURE_TORR)
fd.print_flux_summary(diag, summary)

## 3. Visualizing the flux breakdown

`plot_anode_flux_comparison` (and its loader, `compare_anode_fluxes` -- see
section 7) is built for comparing several simulations side by side, but
works just as well as a single-case bar chart: one bar per panel (H,
electron, ion flux).

In [ ]:
fig, axes = fd.plot_anode_flux_comparison([summary], labels=['This run'], figsize=(9, 5))
plt.show()

## 4. Cross-check against `J_w`

`J_w` is a second, independently-normalized wall-current diagnostic (see the
module docstring). Where it was recorded, this is a quick sanity check on
the EDF-based flux above -- expect agreement to within about 10%, not exact
(it's a different sampling method). `undercount_factor=1.0` here since
`J_w` reflects the particles WarpX actually recorded, before the anode
boundary-scraping bug-correction.

In [ ]:
q_e = 1.602176634e-19

if 'J_w' in diag.ta_fields:
    fluxes = fd.all_charged_species_fluxes(diag, undercount_factor=1.0)
    net_flux = fluxes['electrons'] - sum(f for s, f in fluxes.items() if s != 'electrons')
    predicted_current = q_e * net_flux
    measured_current = fd.wall_current_from_J_w(diag)
    print(f'Predicted q_e*(e - i) flux: {predicted_current:.2f} A/m^2')
    print(f'J_w:                        {measured_current:.2f} A/m^2')
    print(f'Ratio:                      {predicted_current / measured_current:.3f}')
else:
    print('J_w not recorded for this simulation -- skipping cross-check.')

## 5. H-radical density profile

`solve_h_density_profile` is the diffusion solve behind the H flux above;
plotting it directly shows the full spatial profile, not just its boundary
value.

In [ ]:
n_H = fd.solve_h_density_profile(diag, pressure_torr=PRESSURE_TORR)

fig, ax = plt.subplots(dpi=130)
ax.plot(diag.cells, n_H)
ax.set_xlabel('z [m]')
ax.set_ylabel(r'$n_H$ [m$^{-3}$]')
ax.set_title('Atomic Hydrogen Density Profile')
ax.margins(x=0)
ax.set_ylim(0)
plt.show()

## 6. Saving results

Handy for tabulating across runs, or reloading later without recomputing.

In [ ]:
with open('flux_results.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)

print('Saved flux results to flux_results.json')

## 7. Comparing across multiple runs (e.g. a voltage sweep)

Everything above also works with a *list* of cases, which is where
`compare_anode_fluxes` earns its keep -- e.g. comparing `diags` runs across
several RF voltages or discharge lengths (see
`Aug_2026/rf_dc_paper/analysis.ipynb`/`analysis_length.ipynb` for larger,
real examples, including grouped multi-length comparisons). That needs more
than one directory to be meaningful, so it's left here as a pattern to copy
rather than a cell to run against just `diags`:

```python
sweep_dirs = ['diags_RF50', 'diags_RF100', 'diags_RF150', 'diags_RF200']

fig, axes, results = fd.compare_anode_fluxes(
    sweep_dirs,
    pressure_torr=PRESSURE_TORR,
    labels=['50 V', '100 V', '150 V', '200 V'],
)
plt.show()
```